# IRIS-X ARC-AGI-2 prompt-pilot baseline 001
Private, Internet-disabled, competition-mounted L4 x4 execution authorized by Decision 0016. This notebook runs exactly the frozen 384 baseline jobs with one persistent model worker per GPU, no warm-up generation, retry, reflection, or competition submission.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import torch

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['PYTHONUNBUFFERED'] = '1'
base_root = Path('/kaggle/input/irisx-arc-agi-2-trial-002-smoke-001')
runtime_root = Path('/kaggle/input/irisx-arc-agi-2-prompt-pilot-baseline-001')
competition_challenges = Path('/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json')
outbound = runtime_root / 'prompt-pilot-baseline-001.zip'
runtime_wheel = runtime_root / 'irisx-0.2.0-py3-none-any.whl'
model_root = base_root
EXPECTED_OUTBOUND_SHA256 = '__IRISX_OUTBOUND_SHA256__'
EXPECTED_RUNTIME_WHEEL_SHA256 = '__IRISX_RUNTIME_WHEEL_SHA256__'
if not all(path.is_file() for path in (competition_challenges, outbound, runtime_wheel, model_root / 'config.json', model_root / 'model.safetensors.index.json', base_root / 'transformers-5.8.1-py3-none-any.whl')):
    raise RuntimeError('one or more explicitly bound Kaggle input paths are missing')
if hashlib.sha256(outbound.read_bytes()).hexdigest() != EXPECTED_OUTBOUND_SHA256:
    raise RuntimeError('outbound digest differs from technical preflight')
if hashlib.sha256(runtime_wheel.read_bytes()).hexdigest() != EXPECTED_RUNTIME_WHEEL_SHA256:
    raise RuntimeError('runtime wheel digest differs from technical preflight')
hardware = []
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    hardware.append({'index': index, 'name': props.name, 'memory_mib': int(props.total_memory // (1024 * 1024)), 'uuid': str(getattr(props, 'uuid', 'unknown'))})
smi = subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,uuid,pci.bus_id', '--format=csv,noheader,nounits'], capture_output=True, text=True, check=False, timeout=30)
preflight = {'run_id': 'prompt-pilot-baseline-001', 'torch_cuda_device_count': torch.cuda.device_count(), 'gpus': hardware, 'nvidia_smi_exit_code': smi.returncode, 'nvidia_smi_rows': [line.strip() for line in smi.stdout.splitlines() if line.strip()]}
Path('/kaggle/working/notebook-preflight.json').write_text(json.dumps(preflight, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps({'event': 'notebook_hardware_preflight', **preflight}, sort_keys=True), flush=True)
if torch.cuda.device_count() != 4 or len(hardware) != 4 or len({item['uuid'] for item in hardware}) != 4 or any('L4' not in item['name'] or item['memory_mib'] < 22000 for item in hardware) or smi.returncode != 0 or len(preflight['nvidia_smi_rows']) != 4:
    raise RuntimeError(f'four-L4 notebook preflight failed: {preflight}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(base_root), '--upgrade', '--no-deps', 'transformers==5.8.1', 'jinja2==3.1.6'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--force-reinstall', '--no-deps', str(runtime_wheel)], check=True)
command = [sys.executable, '-u', '-m', 'irisx.kaggle_runner_prompt_pilot', '--outbound', str(outbound), '--model-root', str(model_root), '--competition-challenges', str(competition_challenges), '--output', '/kaggle/working/result.zip', '--checkpoint-output', '/kaggle/working/checkpoint.json', '--progress-output', '/kaggle/working/progress.jsonl']
subprocess.run(command, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
